In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from scipy.signal import savgol_filter

# Konfigurasi path direktori basis data mentah
DATASET_DIR = "dataset/"

# Pemetaan ulang skenario biner berdasarkan batasan wilayah dalam ruangan (Sub-bab 1.3)
SCENARIOS = {
    "ruangan_kosong.csv": 0,  # Kelas 0: Empty (Kondisi dasar ruangan kosong)
    "hambatan_nlos.csv": 0,   # Kelas 0: Empty (Orang di luar ruangan dianggap tidak menempati ruangan)
    "orang_diam.csv": 1,      # Kelas 1: Occupied (Subjek berada di dalam ruangan)
    "orang_bergerak.csv": 1,  # Kelas 1: Occupied (Subjek berada di dalam ruangan)
    "posisi_dekat.csv": 1,    # Kelas 1: Occupied (Subjek berada di dalam ruangan)
    "posisi_jauh.csv": 1      # Kelas 1: Occupied (Subjek berada di dalam ruangan)
}

In [3]:
def parse_csi_amplitude(line):
    """
    Sub-bab 3.2.1: Feature Selection.
    Mengekstrak komponen amplitudo murni dari nilai kompleks subcarrier CSI.
    """
    try:
        start = line.find('[')
        end = line.rfind(']')
        if start != -1 and end != -1 and end > start:
            csi_string = line[start+1:end]
            csi_raw = np.fromstring(csi_string, sep=' ', dtype=np.float32)
            if len(csi_raw) % 2 == 0:
                real = csi_raw[0::2]
                imag = csi_raw[1::2]
                return np.hypot(real, imag)
    except Exception:
        return None
    return None

def handle_missing_values(array_data):
    """
    Sub-bab 3.2.3: Missing Value Handling.
    Restorasi data akibat packet loss menggunakan metode interpolasi linier.
    """
    df_temp = pd.DataFrame(array_data)
    df_temp = df_temp.interpolate(method='linear', axis=0).bfill()
    return df_temp.to_numpy()

def hampel_filter_mad(array_data, window_size=7, n_sigmas=3):
    """
    Sub-bab 3.2.2: Data Outlier Removal.
    Mereduksi pencilan menggunakan Hampel Filter berbasis Median Absolute Deviation (MAD).
    """
    n = len(array_data)
    output_data = array_data.copy()
    k = window_size // 2
    
    for col in range(array_data.shape[1]):
        for i in range(k, n - k):
            window = array_data[i - k : i + k + 1, col]
            median = np.median(window)
            mad = np.median(np.abs(window - median))
            threshold = n_sigmas * 1.4826 * mad
            
            if np.abs(array_data[i, col] - median) > threshold:
                output_data[i, col] = median
    return output_data

def apply_savitzky_golay(array_data, window_length=11, polyorder=2):
    """
    Sub-bab 3.2.4: Filtering dan Smoothing.
    Menghilangkan noise frekuensi tinggi menggunakan filter polinomial Savitzky-Golay.
    """
    return savgol_filter(array_data, window_length=window_length, polyorder=polyorder, axis=0)

In [4]:
all_cleaned_features = []
all_labels = []

print("=== MEMULAI PIPELINE PRA-PEMROSESAN DATASET ===")

for filename, label in SCENARIOS.items():
    file_path = os.path.join(DATASET_DIR, filename)
    if not os.path.exists(file_path):
        print(f"[WARNING] Berkas {filename} tidak ditemukan. Dilewati.")
        continue
        
    print(f"[PROCESSING] Mengekstrak dan memfilter berkas: {filename}...")
    file_features = []
    
    with open(file_path, 'r', errors='ignore') as f:
        for line in f:
            if "CSI_DATA" in line and "[" in line and "]" in line:
                amplitude = parse_csi_amplitude(line)
                if amplitude is not None and len(amplitude) == 64:
                    file_features.append(amplitude)
                    
    if len(file_features) == 0:
        continue
        
    # Konversi data ke format array untuk pemrosesan sinyal digital
    file_features = np.array(file_features)
    
    # Penerapan rantai filter secara berurutan sesuai metodologi penelitian
    file_features = handle_missing_values(file_features)
    file_features = hampel_filter_mad(file_features, window_size=7)
    file_features = apply_savitzky_golay(file_features, window_length=11)
    
    for feat in file_features:
        all_cleaned_features.append(feat)
        all_labels.append(label)

# Deklarasi variabel matriks global hasil pembersihan
X_cleaned = np.array(all_cleaned_features)
y_cleaned = np.array(all_labels)

print("\n=== TAHAP PEMBERSIHAN SELESAI ===")
print(f"Matriks Fitur Terpilih Shape: {X_cleaned.shape}")

=== MEMULAI PIPELINE PRA-PEMROSESAN DATASET ===
[PROCESSING] Mengekstrak dan memfilter berkas: ruangan_kosong.csv...
[PROCESSING] Mengekstrak dan memfilter berkas: hambatan_nlos.csv...
[PROCESSING] Mengekstrak dan memfilter berkas: orang_diam.csv...
[PROCESSING] Mengekstrak dan memfilter berkas: orang_bergerak.csv...
[PROCESSING] Mengekstrak dan memfilter berkas: posisi_dekat.csv...
[PROCESSING] Mengekstrak dan memfilter berkas: posisi_jauh.csv...

=== TAHAP PEMBERSIHAN SELESAI ===
Matriks Fitur Terpilih Shape: (11893, 64)


In [5]:
# Sub-bab 3.2.5: Data Transformation - Min-Max Scaling ke rentang [0, 1]
min_val = np.min(X_cleaned, axis=0)
max_val = np.max(X_cleaned, axis=0)
max_val[max_val == min_val] += 1e-8  # Proteksi pembagian dengan nol (zero division)
X_scaled = (X_cleaned - min_val) / (max_val - min_val)

# Sub-bab 3.2.5: Pembentukan Struktur Tensor 3D (Sliding Window)
TIME_STEPS = 20
STEP_SIZE = 5

X_windows = []
y_windows = []

for label_idx in [0, 1]:
    indices = np.where(y_cleaned == label_idx)[0]
    X_subset = X_scaled[indices]
    
    for i in range(0, len(X_subset) - TIME_STEPS, STEP_SIZE):
        window = X_subset[i : i + TIME_STEPS]
        X_windows.append(window)
        y_windows.append(label_idx)

X_3D = np.array(X_windows)
y_3D = np.array(y_windows)

# Sub-bab 3.2.6: Data Balancing melalui Undersampling kelas mayoritas
idx_class_0 = np.where(y_3D == 0)[0]
idx_class_1 = np.where(y_3D == 1)[0]

min_samples = min(len(idx_class_0), len(idx_class_1))
balanced_idx = np.concatenate([idx_class_0[:min_samples], idx_class_1[:min_samples]])

X_balanced = X_3D[balanced_idx]
y_balanced = y_3D[balanced_idx]

# Pembagian data uji (testing data) dan data latih (training data) secara Stratified
X_train, X_test, y_train, y_test = train_test_split(
    X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced
)

print(f"\n=== STRUKTUR AKHIR DATASET SIAP PAKAI ===")
print(f"X_train Shape (Tensor 3D): {X_train.shape}")
print(f"X_test Shape  (Tensor 3D): {X_test.shape}")

# Ekspor data bersih ke format biner eksternal (.npy)
np.save("X_train.npy", X_train)
np.save("X_test.npy", X_test)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)
print("\n[SUCCESS] Proses pra-pemrosesan selesai. Seluruh berkas .npy berhasil diekspor.")


=== STRUKTUR AKHIR DATASET SIAP PAKAI ===
X_train Shape (Tensor 3D): (1238, 20, 64)
X_test Shape  (Tensor 3D): (310, 20, 64)

[SUCCESS] Proses pra-pemrosesan selesai. Seluruh berkas .npy berhasil diekspor.


In [6]:
import numpy as np
print(f"Mentah Kosong (Kelas 0): {np.sum(y_cleaned == 0)}")
print(f"Mentah Terisi (Kelas 1): {np.sum(y_cleaned == 1)}")

Mentah Kosong (Kelas 0): 3890
Mentah Terisi (Kelas 1): 8003


In [7]:
# Simpan nilai min dan max global hasil kalkulasi dataset latih
np.save('global_min_csi.npy', min_val) # sesuaikan dengan nama variabel min lu
np.save('global_max_csi.npy', max_val) # sesuaikan dengan nama variabel max lu
print("Nilai parameter scaling berhasil diamankan!")

Nilai parameter scaling berhasil diamankan!
